In [1]:

import random

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

torch.manual_seed(42)
random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

La partición se hace en dos pasos
 Primero se separa train con 80, después ese 20 restante se divide en partes iguales para obtener val y test con un 10 por ciento para cada una.

In [2]:
dataset = fetch_california_housing()
x = dataset.data
y = dataset.target
feature_names = dataset.feature_names

x_train, x_temp, y_train, y_temp = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42
)

x_val, x_test, y_val, y_test = train_test_split(
    x_temp,
    y_temp,
    test_size=0.5,
    random_state=42
)

len(x_train), len(x_val), len(x_test), feature_names

(16512,
 2064,
 2064,
 ['MedInc',
  'HouseAge',
  'AveRooms',
  'AveBedrms',
  'Population',
  'AveOccup',
  'Latitude',
  'Longitude'])

In [3]:
x_train = torch.tensor(x_train, dtype=torch.float32)
x_val = torch.tensor(x_val, dtype=torch.float32)
x_test = torch.tensor(x_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
y_val = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)
y_test = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

lat_idx = feature_names.index("Latitude")
lon_idx = feature_names.index("Longitude")
geo_idx = [lat_idx, lon_idx]
num_idx = [i for i in range(len(feature_names)) if i not in geo_idx]

x_train.shape, y_train.shape, num_idx, geo_idx

(torch.Size([16512, 8]), torch.Size([16512, 1]), [0, 1, 2, 3, 4, 5], [6, 7])

 nn.Module el módulo siguiente aprende sus estadísticas con train y las guarda como buffers y luego hace standard scaling sobre esas mismas columnas.

In [8]:
class HousingPreprocessor(nn.Module):
    def __init__(self, num_idx, total_features):
        super().__init__()
        self.num_idx = torch.tensor(num_idx, dtype=torch.long)
        self.total_features = total_features
        self.register_buffer("lower_bounds", torch.zeros(len(num_idx)))
        self.register_buffer("upper_bounds", torch.zeros(len(num_idx)))
        self.register_buffer("means", torch.zeros(len(num_idx)))
        self.register_buffer("stds", torch.ones(len(num_idx)))

    def fit(self, x):
        selected = x[:, self.num_idx]
        q1 = torch.quantile(selected, 0.25, dim=0)
        q3 = torch.quantile(selected, 0.75, dim=0)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        clipped = torch.clamp(selected, min=lower, max=upper)
        means = clipped.mean(dim=0)
        stds = clipped.std(dim=0, unbiased=False)
        stds = torch.where(stds < 1e-6, torch.ones_like(stds), stds)

        self.lower_bounds.copy_(lower)
        self.upper_bounds.copy_(upper)
        self.means.copy_(means)
        self.stds.copy_(stds)
        return self

    def forward(self, x):
        x = x.clone()
        selected = x[:, self.num_idx]
        selected = torch.maximum(selected, self.lower_bounds)
        selected = torch.minimum(selected, self.upper_bounds)
        selected = (selected - self.means) / self.stds
        x[:, self.num_idx] = selected
        return x